# Changes to unit selection between scenarios

In the 10 minute bias Taunton and Worcester maps, some LSOA appear in the Taunton and Worcester catchment areas when those units provide MT, even though in the "base" case their nearest unit offers MT.

This is because these LSOA are for example 6 minutes closer to Taunton than Bristol. When Taunton does not offer MT and there is a 10 minute MT unit bias, Bristol is picked. When Taunton offers MT, the bias is irrelevant and the original nearest unit Taunton is picked again.

Prove this here by looking at LSOA-level results.

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas
import matplotlib.pyplot as plt

In [2]:
dir_data = os.path.join('..', 'data')
dir_output = os.path.join('..', 'output')
# Scenario results:
file_nobias = 'results_full_comparison_taunworc_base.csv'
file_bias10 = 'results_full_comparison_taunworc_base_bias10min.csv'
file_bias15 = 'results_full_comparison_taunworc_base_bias15min.csv'

## Load data

Load LSOA-level geography (unit selection and travel times). Use the files where the base and Taunton & Worcester scenarios are gathered. Load in the data for no travel bias for MT units (=0.1 minutes) and the 10 minute bias (=10.1 minutes).

In [3]:
p = os.path.join('..','output', file_nobias)
df_lsoa_nobias = pd.read_csv(p, index_col=0)

p = os.path.join('..','output', file_bias10)
df_lsoa_bias10 = pd.read_csv(p, index_col=0)

p = os.path.join('..','output', file_bias15)
df_lsoa_bias15 = pd.read_csv(p, index_col=0)

In [4]:
dict_df_lsoa = {
    'no_bias': df_lsoa_nobias,
    'bias10min': df_lsoa_bias10,
    'bias15min': df_lsoa_bias15,
}

In [5]:
print(df_lsoa_nobias.columns)

Index(['area_lower', 'Admissions_2025_Est', 'nearest_ivt_unit_base',
       'nearest_ivt_time_base', 'transfer_unit_base', 'transfer_required_base',
       'drip_ship_mt_time_base', 'drip_ship_mt_travel_time_base',
       'drip_ship_mt_travel_time_with_delay_base', 'admissions_mt_base',
       'transfers_mt_base', 'admissions_mt_transferloss0!67_base',
       'transfers_mt_transferloss0!67_base',
       'admissions_weighted_drip_ship_mrs_0-2_base',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'admissions_weighted_drip_ship_mrs_0-2_transferloss0!67_base',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_base',
       'drip_ship_mrs_0-2_base', 'lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'drip_ship_mrs_0-2_transferloss0!67_base',
       'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_base',
       'nearest_ivt_unit_taunworc', 'nearest_ivt_time_taunworc',
       'transfer_unit_taunworc', 'transfer_required_taunworc',
       'drip_ship_mt_ti

## Change of nearest unit due to scenario

Pick out any LSOA where the nearest unit offers MT in both scenarios (transfer unit is the same as the nearest unit), but a different unit is chosen between scenarios (nearest unit base is different from nearest unit taunworc).

Calculate this for all of the MT travel bias options.

In [6]:
# Display only these columns:
cols = [
    'area_lower',  # for plots
    'Admissions_2025_Est',
    'nearest_ivt_unit_base',
    'nearest_ivt_time_base',
    'drip_ship_mt_travel_time_base',
    'nearest_ivt_unit_taunworc',
    'nearest_ivt_time_taunworc',
    'drip_ship_mt_travel_time_taunworc',
]

# Unit changes when base unit provides MT:
dict_lsoa_mt_first = {}
for s_bias, df in dict_df_lsoa.items():
    # Display only rows that meet these conditions:
    m = (
        # Base nearest unit offers MT:
        (df['nearest_ivt_unit_base'] == df['transfer_unit_base']) &
        # Nearest unit changes in Taunton & Worcester scenario:
        (df['nearest_ivt_unit_base'] != df['nearest_ivt_unit_taunworc'])
    )
    # Limit the full dataframe to these rows and columns:
    df_here = df.loc[m, cols].copy()
    # Calculate the extra travel time in the base case:
    df_here['time_diff'] = (df_here['nearest_ivt_time_base'] -
                            df_here['nearest_ivt_time_taunworc'])
    # Store result:
    dict_lsoa_mt_first[s_bias] = df_here.copy()

# Any unit change:
dict_lsoa_change = {}
for s_bias, df in dict_df_lsoa.items():
    # Display only rows that meet these conditions:
    m = (
        # Nearest unit changes in Taunton & Worcester scenario:
        (df['nearest_ivt_unit_base'] != df['nearest_ivt_unit_taunworc'])
    )
    # Limit the full dataframe to these rows and columns:
    df_here = df.loc[m, cols].copy()  
    # Calculate the extra travel time in the base case:
    df_here['time_diff'] = (df_here['nearest_ivt_time_base'] -
                            df_here['nearest_ivt_time_taunworc'])
    # Store result:
    dict_lsoa_change[s_bias] = df_here.copy()

Show preview of results:

In [7]:
dict_lsoa_mt_first['bias10min'].head(5)

,area_lower,Admissions_2025_Est,nearest_ivt_unit_base,nearest_ivt_time_base,drip_ship_mt_travel_time_base,nearest_ivt_unit_taunworc,nearest_ivt_time_taunworc,drip_ship_mt_travel_time_taunworc,time_diff
LSOA,,,,,,,,,
BROMSGROVE010A,Bromsgrove 010A,2.24110,B152TH,19,19,WR51DD,18,18,1
BROMSGROVE010B,Bromsgrove 010B,6.34972,B152TH,20,20,WR51DD,16,16,4
BROMSGROVE010C,Bromsgrove 010C,0.37348,B152TH,20,20,WR51DD,16,16,4
BROMSGROVE010D,Bromsgrove 010D,3.73513,B152TH,19,19,WR51DD,17,17,2
BROMSGROVE011A,Bromsgrove 011A,1.86762,B152TH,20,20,WR51DD,15,15,5


In [8]:
dict_lsoa_change['bias10min'].head(5)

,area_lower,Admissions_2025_Est,nearest_ivt_unit_base,nearest_ivt_time_base,drip_ship_mt_travel_time_base,nearest_ivt_unit_taunworc,nearest_ivt_time_taunworc,drip_ship_mt_travel_time_taunworc,time_diff
LSOA,,,,,,,,,
BROMSGROVE010A,Bromsgrove 010A,2.24110,B152TH,19,19,WR51DD,18,18,1
BROMSGROVE010B,Bromsgrove 010B,6.34972,B152TH,20,20,WR51DD,16,16,4
BROMSGROVE010C,Bromsgrove 010C,0.37348,B152TH,20,20,WR51DD,16,16,4
BROMSGROVE010D,Bromsgrove 010D,3.73513,B152TH,19,19,WR51DD,17,17,2
BROMSGROVE011A,Bromsgrove 011A,1.86762,B152TH,20,20,WR51DD,15,15,5


Save these to file:

In [9]:
for label, df in dict_lsoa_mt_first.items():
    f = f'lsoa_change_unit_mt_availability_{label}.csv'
    p = os.path.join('output', f)
    df.to_csv(p, index=True)

for label, df in dict_lsoa_change.items():
    f = f'lsoa_change_unit_{label}.csv'
    p = os.path.join('output', f)
    df.to_csv(p, index=True)

### Display numbers of LSOA with each travel time difference

The values in these DataFrames are the numbers of LSOA that have that difference in travel time between the units in the base scenario and Taunton+Worcester scenario. The column names are (nearest unit in base, nearest unit in taunworc).

In [10]:
def make_df_time_counts(df, cols_units):
    dfs_to_concat = []

    for unit_combos, df_here in df.groupby(cols_units):
        s = df_here['time_diff'].value_counts().sort_index()
        s.name = unit_combos
        dfs_to_concat.append(s)

    df_time_diff_counts = pd.concat(dfs_to_concat, axis='columns')
    # Label MultiIndex column levels:
    df_time_diff_counts.columns.names = cols_units
    # Sort times:
    df_time_diff_counts = df_time_diff_counts.sort_index()
    # Fill missing times with zeros:
    df_time_diff_counts = df_time_diff_counts.fillna(0).convert_dtypes()

    return df_time_diff_counts

In [11]:
cols_units = ['nearest_ivt_unit_base', 'nearest_ivt_unit_taunworc']

for s_bias, df in dict_lsoa_mt_first.items():
    print(s_bias)
    df_time_diff_counts = make_df_time_counts(df, cols_units)
    print('Count of LSOA in each time difference bin:')
    display(df_time_diff_counts)
    # Totals:
    print('Total LSOA:')
    display(df_time_diff_counts.sum(axis='rows'))
    print('')

no_bias
Count of LSOA in each time difference bin:


nearest_ivt_unit_base,BS105NB,CV22DX
nearest_ivt_unit_taunworc,TA15DA,WR51DD
time_diff,,
0,1,2


Total LSOA:


nearest_ivt_unit_base  nearest_ivt_unit_taunworc
BS105NB                TA15DA                       1
CV22DX                 WR51DD                       2
dtype: Int64


bias10min
Count of LSOA in each time difference bin:


nearest_ivt_unit_base,B152TH,BS105NB,CV22DX,OX39DU
nearest_ivt_unit_taunworc,WR51DD,TA15DA,WR51DD,WR51DD
time_diff,,,,
0,0,1,2,0
1,5,2,2,0
2,13,0,0,0
3,18,3,0,0
4,9,0,0,0
5,11,1,2,1
6,8,3,0,0
7,12,4,0,0


... (2 outputs left out)


bias15min
Count of LSOA in each time difference bin:


nearest_ivt_unit_base,B152TH,BS105NB,CV22DX,OX39DU
nearest_ivt_unit_taunworc,WR51DD,TA15DA,WR51DD,WR51DD
time_diff,,,,
0,0,1,2,0
1,5,2,2,0
2,15,0,0,1
3,20,3,0,0
4,9,2,0,0
5,11,1,2,1
6,10,3,0,1
7,12,4,0,1


Total LSOA:


nearest_ivt_unit_base  nearest_ivt_unit_taunworc
B152TH                 WR51DD                       124
BS105NB                TA15DA                        39
CV22DX                 WR51DD                        11
OX39DU                 WR51DD                         4
dtype: Int64

All of these results tables are the same up to the travel time bias. For example, the table for the 15 minute bias has the same data up to 10 minutes' time difference as the 10 minutes bias.

### Check: changes with same times

With zero travel time bias, the only LSOA that changed units did so because a new unit with MT became available at no extra travel time.

Check whether this is the case for any of the time bias scenarios. Make copies of the above results where only rows with a time difference of zero are kept.

In [12]:
dict_lsoa_mt_first_zero = {}
for s_bias, df_b in dict_lsoa_mt_first.items():
    df = df_b[df_b['time_diff'] < 1]
    dict_lsoa_mt_first_zero[s_bias] = df.copy()

If the two scenarios are identical, then the following cell will print True:

In [13]:
b1 = dict_lsoa_mt_first_zero['no_bias'].equals(
    dict_lsoa_mt_first_zero['bias10min'])
b2 = dict_lsoa_mt_first_zero['no_bias'].equals(
    dict_lsoa_mt_first_zero['bias15min'])
print(b1, b2)

True True


The results are the same for all of the MT travel bias options.

Show the no-bias options again:

In [14]:
display(dict_lsoa_mt_first_zero['no_bias'])

,area_lower,Admissions_2025_Est,nearest_ivt_unit_base,nearest_ivt_time_base,drip_ship_mt_travel_time_base,nearest_ivt_unit_taunworc,nearest_ivt_time_taunworc,drip_ship_mt_travel_time_taunworc,time_diff
LSOA,,,,,,,,,
NORTHSOMERSET024B,North Somerset 024B,2.98807,BS105NB,34,34,TA15DA,34,34,0
STRATFORD-ON-AVON011A,Stratford-on-Avon 011A,1.49403,CV22DX,29,29,WR51DD,29,29,0
STRATFORD-ON-AVON011B,Stratford-on-Avon 011B,1.49403,CV22DX,27,27,WR51DD,27,27,0


Both of the Stratford-upon-Avon LSOA have the same travel time to both Coventry (CV22DX) and Worcester (WR51DD). In the base case, only Coventry provides MT and so the LSOA pick that. In the Taunton & Worcester case, both Coventry and Worcester provide MT so the LSOA pick the one that is closer in terms of distance, which is Worcester.

Similarly for the North Somerset LSOA, the travel time is the same to both Bristol (BS105NB) and Taunton (TA15DA), but when MT is available at both, Taunton is slightly closer in terms of distance.

### Repeat for units that change when nearest does not offer MT

In [15]:
cols_units = ['nearest_ivt_unit_base', 'nearest_ivt_unit_taunworc']

for s_bias, df in dict_lsoa_change.items():
    print(s_bias)
    df_time_diff_counts = make_df_time_counts(df, cols_units)
    # print('Count of LSOA in each time difference bin:')
    # display(df_time_diff_counts)
    # Too many counts and times to be of much use.
    # Totals:
    print('Total LSOA:')
    display(df_time_diff_counts.sum(axis='rows'))
    print('')

no_bias
Total LSOA:


nearest_ivt_unit_base  nearest_ivt_unit_taunworc
BS105NB                TA15DA                       1
CV22DX                 WR51DD                       2
DT12JY                 TA15DA                       1
DY12HQ                 WR51DD                       1
dtype: Int64


bias10min
Total LSOA:


nearest_ivt_unit_base  nearest_ivt_unit_taunworc
B152TH                 WR51DD                       89
BA13NG                 TA15DA                        2
BS105NB                TA15DA                       16
CV22DX                 WR51DD                        9
DT12JY                 TA15DA                       47
DY12HQ                 WR51DD                        3
EX25DW                 TA15DA                       25
EX314JB                TA15DA                        3
GL537AN                WR51DD                       26
HR12ER                 WR51DD                       19
OX39DU                 WR51DD                        1
SP28BJ                 TA15DA                        5
TF16TF                 WR51DD                        3
dtype: Int64


bias15min
Total LSOA:


nearest_ivt_unit_base  nearest_ivt_unit_taunworc
B152TH                 WR51DD                       124
BS105NB                TA15DA                        39
CV22DX                 WR51DD                        11
DT12JY                 TA15DA                        61
EX25DW                 TA15DA                        37
EX314JB                TA15DA                         4
GL537AN                WR51DD                        32
HR12ER                 WR51DD                        33
OX39DU                 WR51DD                         4
SP28BJ                 TA15DA                         9
TF16TF                 WR51DD                         1
dtype: Int64